In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col 
from pyspark.sql.functions import broadcast


In [2]:
# create the spark session
spark = SparkSession.builder.appName("spark_fund_homework").getOrCreate()

25/08/15 05:25:58 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [44]:
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, TimestampType, IntegerType, DoubleType


In [40]:
matches_schema = StructType([
    StructField("match_id", StringType(), True),
    StructField("mapid", StringType(), True),
    StructField("is_team_game", BooleanType(), True),
    StructField("playlist_id", StringType(), True),
    StructField("game_variant_id", StringType(), True),
    StructField("is_match_over", BooleanType(), True),
    StructField("completion_date", TimestampType(), True),
    StructField("match_duration", IntegerType(), True),
    StructField("game_mode", StringType(), True),
    StructField("map_variant_id", StringType(), True)
])

In [46]:
match_details_schema = StructType([
    StructField("match_id", StringType(), True),
    StructField("player_gamertag", StringType(), True),
    StructField("previous_spartan_rank", IntegerType(), True),
    StructField("spartan_rank", IntegerType(), True),
    StructField("previous_total_xp", IntegerType(), True),
    StructField("total_xp", IntegerType(), True),
    StructField("previous_csr_tier", IntegerType(), True),
    StructField("previous_csr_designation", IntegerType(), True),
    StructField("previous_csr", IntegerType(), True),
    StructField("previous_csr_percent_to_next_tier", IntegerType(), True),
    StructField("previous_csr_rank", IntegerType(), True),
    StructField("current_csr_tier", IntegerType(), True),
    StructField("current_csr_designation", IntegerType(), True),
    StructField("current_csr", IntegerType(), True),
    StructField("current_csr_percent_to_next_tier", IntegerType(), True),
    StructField("current_csr_rank", IntegerType(), True),
    StructField("player_rank_on_team", IntegerType(), True),
    StructField("player_finished", BooleanType(), True),
    StructField("player_average_life", StringType(), True),
    StructField("player_total_kills", IntegerType(), True),
    StructField("player_total_headshots", IntegerType(), True),
    StructField("player_total_weapon_damage", DoubleType(), True),
    StructField("player_total_shots_landed", IntegerType(), True),
    StructField("player_total_melee_kills", IntegerType(), True),
    StructField("player_total_melee_damage", DoubleType(), True),
    StructField("player_total_assassinations", IntegerType(), True),
    StructField("player_total_ground_pound_kills", IntegerType(), True),
    StructField("player_total_shoulder_bash_kills", IntegerType(), True),
    StructField("player_total_grenade_damage", DoubleType(), True),
    StructField("player_total_power_weapon_damage", DoubleType(), True),
    StructField("player_total_power_weapon_grabs", IntegerType(), True),
    StructField("player_total_deaths", IntegerType(), True),
    StructField("player_total_assists", IntegerType(), True),
    StructField("player_total_grenade_kills", IntegerType(), True),
    StructField("did_win", BooleanType(), True),
    StructField("team_id", IntegerType(), True)
])


In [47]:
# read data from files 
df_matches = spark.read \
    .option("header", "true") \
    .option("inferSchema", False) \
    .schema(matches_schema) \
    .csv("/home/iceberg/data/matches.csv")

df_maps = spark.read \
    .option("header", "true") \
    .csv("/home/iceberg/data/maps.csv")

df_match_details = spark.read \
    .option("header", "true") \
    .option("inferSchema", False) \
    .schema(match_details_schema) \
    .csv("/home/iceberg/data/match_details.csv")

df_medal_matches_players = spark.read \
    .option("header", "true") \
    .csv("/home/iceberg/data/medals_matches_players.csv")

df_medals = spark.read \
    .option("header", "true") \
    .csv("/home/iceberg/data/medals.csv")

In [42]:
df_matches.show()

+--------------------+--------------------+------------+--------------------+--------------------+-------------+-------------------+--------------+---------+--------------------+
|            match_id|               mapid|is_team_game|         playlist_id|     game_variant_id|is_match_over|    completion_date|match_duration|game_mode|      map_variant_id|
+--------------------+--------------------+------------+--------------------+--------------------+-------------+-------------------+--------------+---------+--------------------+
|11de1a94-8d07-416...|c7edbf0f-f206-11e...|        true|f72e0ef0-7c4a-430...|1e473914-46e4-408...|         true|2016-02-22 00:00:00|          NULL|     NULL|                NULL|
|d3643e71-3e51-43e...|cb914b9e-f206-11e...|       false|d0766624-dbd7-453...|257a305e-4dd3-41f...|         true|2016-02-14 00:00:00|          NULL|     NULL|                NULL|
|d78d2aae-36e4-48a...|c7edbf0f-f206-11e...|        true|f72e0ef0-7c4a-430...|1e473914-46e4-408...|       

In [ ]:
# Disabled automatic broadcast join with `spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")`
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [4]:
# Explicitly broadcast JOINs `medals` and `maps`
df_medals_map = df_medals.join(broadcast(df_maps), how="inner")

In [5]:
df_medals_map.show()

+----------+----------+-----------+----------+------------------+-------------------+------------+-------------+--------------+-----------+----+----------+--------------------+-------------------+--------------------+
|  medal_id|sprite_uri|sprite_left|sprite_top|sprite_sheet_width|sprite_sheet_height|sprite_width|sprite_height|classification|description|name|difficulty|               mapid|               name|         description|
+----------+----------+-----------+----------+------------------+-------------------+------------+-------------+--------------+-----------+----+----------+--------------------+-------------------+--------------------+
|2315448068|      NULL|       NULL|      NULL|              NULL|               NULL|        NULL|         NULL|          NULL|       NULL|NULL|      NULL|c93d708f-f206-11e...|              Urban|Andesia was the c...|
|2315448068|      NULL|       NULL|      NULL|              NULL|               NULL|        NULL|         NULL|          NULL| 

In [6]:
df_medals_map.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastNestedLoopJoin BuildRight, Inner
   :- FileScan csv [medal_id#191,sprite_uri#192,sprite_left#193,sprite_top#194,sprite_sheet_width#195,sprite_sheet_height#196,sprite_width#197,sprite_height#198,classification#199,description#200,name#201,difficulty#202] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/iceberg/data/medals.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<medal_id:string,sprite_uri:string,sprite_left:string,sprite_top:string,sprite_sheet_width:...
   +- BroadcastExchange IdentityBroadcastMode, [plan_id=175]
      +- FileScan csv [mapid#54,name#55,description#56] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/iceberg/data/maps.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<mapid:string,name:string,description:string>




In [33]:
match_details_ddl_sql = """
CREATE TABLE bootcamp.match_details_bucketed (
    match_id STRING,
    player_gamertag STRING,
    previous_spartan_rank STRING,
    spartan_rank STRING,
    previous_total_xp STRING,
    total_xp STRING,
    previous_csr_tier STRING,
    previous_csr_designation STRING,
    previous_csr STRING,
    previous_csr_percent_to_next_tier STRING,
    previous_csr_rank STRING,
    current_csr_tier STRING,
    current_csr_designation STRING,
    current_csr STRING,
    current_csr_percent_to_next_tier STRING,
    current_csr_rank STRING,
    player_rank_on_team STRING,
    player_finished STRING,
    player_average_life STRING,
    player_total_kills STRING,
    player_total_headshots STRING,
    player_total_weapon_damage STRING,
    player_total_shots_landed STRING,
    player_total_melee_kills STRING,
    player_total_melee_damage STRING,
    player_total_assassinations STRING,
    player_total_ground_pound_kills STRING,
    player_total_shoulder_bash_kills STRING,
    player_total_grenade_damage STRING,
    player_total_power_weapon_damage STRING,
    player_total_power_weapon_grabs STRING,
    player_total_deaths STRING,
    player_total_assists STRING,
    player_total_grenade_kills STRING,
    did_win STRING,
    team_id STRING
) USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""

In [48]:
match_details_ddl_sql = """
CREATE TABLE bootcamp.match_details_bucketed (
    match_id STRING,
    player_gamertag STRING,
    previous_spartan_rank INT,
    spartan_rank INT,
    previous_total_xp INT,
    total_xp INT,
    previous_csr_tier INT,
    previous_csr_designation INT,
    previous_csr INT,
    previous_csr_percent_to_next_tier INT,
    previous_csr_rank INT,
    current_csr_tier INT,
    current_csr_designation INT,
    current_csr INT,
    current_csr_percent_to_next_tier INT,
    current_csr_rank INT,
    player_rank_on_team INT,
    player_finished BOOLEAN,
    player_average_life STRING,
    player_total_kills INT,
    player_total_headshots INT,
    player_total_weapon_damage DOUBLE,
    player_total_shots_landed INT,
    player_total_melee_kills INT,
    player_total_melee_damage DOUBLE,
    player_total_assassinations INT,
    player_total_ground_pound_kills INT,
    player_total_shoulder_bash_kills INT,
    player_total_grenade_damage DOUBLE,
    player_total_power_weapon_damage DOUBLE,
    player_total_power_weapon_grabs INT,
    player_total_deaths INT,
    player_total_assists INT,
    player_total_grenade_kills INT,
    did_win BOOLEAN,
    team_id INT
) USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""

In [50]:
spark.sql(match_details_ddl_sql) 

DataFrame[]

In [49]:
spark.sql("DROP TABLE IF EXISTS bootcamp.match_details_bucketed ") 

DataFrame[]

In [11]:
matches_ddl_sql = """
CREATE TABLE bootcamp.matches_bucketed (
    match_id STRING,
    mapid STRING,
    is_team_game BOOLEAN,
    playlist_id STRING,
    game_variant_id STRING,
    is_match_over BOOLEAN,
    completion_date TIMESTAMP,
    match_duration INT,
    game_mode STRING,
    map_variant_id STRING
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""

In [12]:
spark.sql(matches_ddl_sql)

DataFrame[]

In [13]:
medals_match_players_ddl_sql = """
CREATE TABLE bootcamp.medals_match_players_bucketed (
    match_id STRING,
    player_gamertag STRING,
    medal_id STRING,
    count INT
)
USING iceberg
PARTITIONED BY (bucket(16, match_id));
"""

In [14]:
spark.sql(medals_match_players_ddl_sql)

DataFrame[]

In [ ]:
#   - Bucket join `match_details`, `matches`, and `medal_matches_players` on `match_id` with `16` buckets
spark.sql("""
SELECT * 
FROM bootcamp.match_details_bucketed 
""")

In [51]:
df_match_details.write.mode("append") \
.bucketBy(16, "match_id") \
.saveAsTable("bootcamp.match_details_bucketed")

In [43]:
df_matches.write.mode("append") \
.bucketBy(16, "match_id") \
.saveAsTable("bootcamp.matches_bucketed")